# Predictive Analytics for Healthcare: Forecasting Patient Length of Stay in Urban Hospitals

**IDRA Data Science & AI Capstone Project — Project 1**

This notebook contains the complete reproducible workflow for the capstone:
data understanding → cleaning → preprocessing → EDA → statistical analysis → regression modelling → evaluation → findings.

**Target:** `lengthofstay`

**Important leakage decision:** `discharged` is not used as a predictor because it occurs after the admission and is directly tied to the target. `eid` is an identifier and is also excluded. The raw `vdate` is converted into admission-month and admission-weekday features rather than being used as a raw date.


## 1. Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "P_1_LengthOfStay.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


## 2. Initial Data Understanding and Quality Checks

In [ ]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("Duplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T)
print("rcount values:")
display(df["rcount"].value_counts(dropna=False).sort_index())


### Data-quality observations

- The dataset contains 61,680 observations and 28 original columns.
- No missing values are present in the raw file.
- No duplicate rows are present.
- `rcount` contains the category `5+`; this is converted to 5 for modelling.
- One negative glucose value is present. Because glucose cannot be negative, it is treated as an invalid observation and replaced with a missing value so that the modelling pipeline can impute it from the training data.
- Extreme laboratory values are retained unless they are demonstrably invalid; the aim is to avoid removing potentially meaningful clinical observations without justification.


## 3. Cleaning and Feature Engineering

In [ ]:
clean = df.copy()

# Convert readmission count to numeric; cap the top-coded category at 5
clean["rcount_num"] = clean["rcount"].replace({"5+": 5}).astype(int)

# Parse admission date and derive calendar features
clean["admission_date"] = pd.to_datetime(clean["vdate"], format="%m/%d/%Y")
clean["admission_month"] = clean["admission_date"].dt.month
clean["admission_dayofweek"] = clean["admission_date"].dt.dayofweek

# Treat the single negative glucose value as invalid
clean.loc[clean["glucose"] < 0, "glucose"] = np.nan

print("Rows after cleaning:", len(clean))
print("Remaining duplicate rows:", clean.duplicated().sum())
print("Missing values introduced for invalid glucose:", clean["glucose"].isna().sum())


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,4.5))
plt.hist(clean["lengthofstay"], bins=np.arange(.5, 17.6, 1), edgecolor="black")
plt.xlabel("Length of stay (days)")
plt.ylabel("Number of admissions")
plt.title("Distribution of Patient Length of Stay")
plt.show()


In [ ]:
g = clean.groupby("rcount_num")["lengthofstay"].agg(["mean","median"]).reset_index()
x = np.arange(len(g)); w = .36

plt.figure(figsize=(7,4.5))
plt.bar(x-w/2, g["mean"], width=w, label="Mean")
plt.bar(x+w/2, g["median"], width=w, label="Median")
plt.xticks(x, g["rcount_num"])
plt.xlabel("Readmissions in previous 180 days (5+ coded as 5)")
plt.ylabel("Length of stay (days)")
plt.title("Length of Stay by Readmission Count")
plt.legend()
plt.show()


In [ ]:
fac = clean.groupby("facid")["lengthofstay"].mean().sort_values()
plt.figure(figsize=(7,4.5))
plt.bar(fac.index, fac.values)
plt.xlabel("Facility ID")
plt.ylabel("Mean length of stay (days)")
plt.title("Mean Length of Stay by Facility")
plt.show()


In [ ]:
binary = [
    "dialysisrenalendstage","asthma","irondef","pneum",
    "substancedependence","psychologicaldisordermajor","depress",
    "psychother","fibrosisandother","malnutrition","hemo"
]

condition_results = []
for c in binary:
    mean_absent = clean.loc[clean[c] == 0, "lengthofstay"].mean()
    mean_present = clean.loc[clean[c] == 1, "lengthofstay"].mean()
    condition_results.append(
        [c, mean_absent, mean_present, mean_present - mean_absent]
    )

condition_results = pd.DataFrame(
    condition_results,
    columns=["condition","mean_absent","mean_present","difference"]
).sort_values("difference", ascending=False)

display(condition_results)

top = condition_results.head(6).sort_values("difference")
plt.figure(figsize=(8,4.8))
plt.barh(top["condition"], top["difference"])
plt.xlabel("Difference in mean stay (days)")
plt.title("Conditions Associated with Longer Average Stays")
plt.show()


In [ ]:
corr_cols = [
    "rcount_num","hematocrit","neutrophils","sodium","glucose",
    "bloodureanitro","creatinine","bmi","pulse","respiration",
    "secondarydiagnosisnonicd9","lengthofstay"
]
corr = clean[corr_cols].corr()
display(corr["lengthofstay"].sort_values(ascending=False).to_frame("correlation"))

plt.figure(figsize=(8,6))
plt.imshow(corr, aspect="auto", interpolation="nearest")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=70, ha="right", fontsize=8)
plt.yticks(range(len(corr_cols)), corr_cols, fontsize=8)
plt.title("Correlation Matrix of Key Numerical Variables")
plt.tight_layout()
plt.show()


## 5. Statistical Analysis

In [ ]:
stats = clean[[
    "lengthofstay","rcount_num","hematocrit","neutrophils",
    "sodium","glucose","bloodureanitro","creatinine","bmi","pulse",
    "respiration","secondarydiagnosisnonicd9"
]].describe().T

stats["IQR"] = stats["75%"] - stats["25%"]
display(stats[["count","mean","50%","std","min","max","IQR"]].rename(
    columns={"50%":"median"}
))


## 6. Modelling Strategy

Two regression models are used:

1. **Linear Regression** — a transparent baseline.
2. **Random Forest Regressor** — the final nonlinear model, selected because the EDA suggests nonlinear and interaction effects across readmission history, facility, comorbidity indicators and laboratory measurements.

The train-test split is 80:20 with `random_state=42`. Numeric variables are median-imputed and standardized; categorical variables (`gender`, `facid`) are one-hot encoded. The preprocessing is fitted only on the training data through a scikit-learn pipeline to avoid data leakage.


In [ ]:
binary = [
    "dialysisrenalendstage","asthma","irondef","pneum",
    "substancedependence","psychologicaldisordermajor","depress",
    "psychother","fibrosisandother","malnutrition","hemo"
]

numeric = [
    "rcount_num","hematocrit","neutrophils","sodium","glucose",
    "bloodureanitro","creatinine","bmi","pulse","respiration",
    "secondarydiagnosisnonicd9","admission_month","admission_dayofweek"
] + binary

categorical = ["gender","facid"]

X = clean[numeric + categorical]
y = clean["lengthofstay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

linear_model = Pipeline([
    ("pre", preprocessor),
    ("model", LinearRegression())
])

rf_model = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=2,
        max_features="sqrt"
    ))
])

linear_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

pred_linear = linear_model.predict(X_test)
pred_rf = rf_model.predict(X_test)
pred_rf_train = rf_model.predict(X_train)


In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    "Linear Regression": evaluate(y_test, pred_linear),
    "Random Forest": evaluate(y_test, pred_rf)
}).T

display(results.round(4))

print("Random Forest training performance:")
display(pd.DataFrame([evaluate(y_train, pred_rf_train)]).round(4))


In [ ]:
plt.figure(figsize=(7,4.5))
plt.scatter(y_test, pred_rf, s=10, alpha=.25)
mn, mx = min(y_test.min(), pred_rf.min()), max(y_test.max(), pred_rf.max())
plt.plot([mn,mx],[mn,mx], linestyle="--")
plt.xlabel("Actual length of stay (days)")
plt.ylabel("Predicted length of stay (days)")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

residuals = y_test - pred_rf
plt.figure(figsize=(7,4.5))
plt.scatter(pred_rf, residuals, s=10, alpha=.25)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted length of stay (days)")
plt.ylabel("Residual")
plt.title("Random Forest Residual Plot")
plt.show()


In [ ]:
feature_names = rf_model.named_steps["pre"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
)

display(feature_importance.head(15))


## 7. Final Findings

- The target has a mean stay of about 4 days and a median of 4 days.
- Readmission count is the strongest simple numerical association with length of stay (`r ≈ 0.75`).
- Facility differences are substantial: facilities A/B have shorter average stays than C/D/E.
- Several comorbidity indicators show higher average stays when present, especially dialysis/renal end-stage, psychotherapy, fibrosis/other conditions, hemo/blood disorder, malnutrition and pneumonia.
- Random Forest substantially outperforms the linear baseline and explains about 91% of test-set variance, with an MAE of about 0.49 days.
- The model should be treated as a decision-support tool rather than a causal or clinical decision rule.
